# 3.44 — Hyperparameter Search

Hyperparameter search chooses settings outside the learned model — learning rates, regularization strengths, tree depths, trial budgets, and more — by measuring which setting survives validation data best. In this lesson, you will build grid search, random search, Bayesian-style surrogate search, Hyperband-style resource allocation, and validation-noise checks from scratch with NumPy so the decision rule is always visible.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build hyperparameter search one idea at a time. Run each cell in order and read the printed intermediate values — every search method is reduced to arrays, averages, validation scores, and explicit decisions. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, random numbers, and small numerical checks for every search method.
import matplotlib.pyplot as plt  # visualizations for score curves, search traces, and budgets.
np.random.seed(0)  # reproducibility for stochastic search examples.

▶ What you'll see: the only two libraries used in the notebook are loaded, and randomness is fixed.

### 1. A hyperparameter is chosen by validation risk, not training comfort

A learned parameter is fitted from data; a **hyperparameter** is chosen around the fitting procedure. The core rule is

$$\lambda^*=\arg\min_{\lambda\in\Lambda} R_{val}(\hat f_\lambda).$$

The important part is not the symbol $\lambda$; it is the split of responsibilities. Training loss says how well a setting can adapt to the training examples, while validation loss estimates whether that adaptation transfers. A setting that wins on training but loses on validation is usually too flexible for the available data.

In [ ]:
lambdas_w = np.array([0.0, 0.1, 1.0, 10.0])  # four candidate regularization strengths.
train_w = np.array([0.06, 0.09, 0.18, 0.42])  # training losses usually rise as constraint grows.
val_w = np.array([0.44, 0.31, 0.24, 0.36])  # validation risk has a best middle setting.
best_idx_w = int(np.argmin(val_w))  # choose by validation loss, not by the smallest training loss.
print("best lambda:", lambdas_w[best_idx_w])
print("best validation risk:", val_w[best_idx_w])
assert lambdas_w[best_idx_w] == 1.0
assert val_w[best_idx_w] == 0.24

▶ What you'll see: `lambda=1.0` wins even though `lambda=0.0` has the lowest training loss.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(lambdas_w, train_w, marker="o", label="train loss")
plt.plot(lambdas_w, val_w, marker="o", label="validation loss")
plt.xscale("symlog")
plt.axvline(lambdas_w[best_idx_w], color="crimson", linestyle="--", label="chosen λ")
plt.title("1: choose the validation winner")
plt.xlabel("regularization λ")
plt.ylabel("loss")
plt.legend()
plt.show()

▶ What you'll see: the train curve likes flexibility, while the validation curve bottoms out at a more stable middle value.

*Why it's done this way:* training loss is biased toward settings with more freedom because those settings can contort themselves around the observed sample. Validation risk is a proxy for future risk, so the argmin is taken over validation scores. The math is an empirical version of generalization: choose the setting whose fitted model has the smallest estimated out-of-sample loss.

### 2. The decision score can include cost, penalty, or operational complexity

The lesson's toy arithmetic starts from three per-example losses: `0.268`, `0.135`, and `0.437`. Their average is the empirical risk fragment, but the setting also carries a cost of `0.070`. The selection score is therefore not merely the raw average; it is the full quantity the method promises to minimize.

In [ ]:
losses_w = np.array([0.268, 0.135, 0.437])  # verified per-example validation losses.
raw_risk_w = float(np.mean(losses_w))  # empirical average over examples.
cost_w = 0.070  # complexity, regularization, or operational cost for this setting.
score_w = raw_risk_w + cost_w  # full decision score.
print("raw validation risk:", round(raw_risk_w, 3))
print("cost:", round(cost_w, 3))
print("decision score:", round(score_w, 3))
assert round(raw_risk_w, 3) == 0.280
assert round(score_w, 3) == 0.350

▶ What you'll see: the raw average is `0.280`, but the score used for selection is `0.350`.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["raw risk", "cost", "full score"], [raw_risk_w, cost_w, score_w], color=["steelblue", "orange", "seagreen"])
plt.title("2: score = validation risk + cost")
plt.ylabel("score component")
plt.show()

▶ What you'll see: the full score is visibly larger than the raw validation fragment because it includes the extra cost term.

*Why it's done this way:* adding cost is a compact way to encode a preference for simpler, cheaper, or more regularized settings. If two settings have similar validation risk, the one that spends less flexibility is usually safer. Mathematically, the score changes the optimization target from “fit this validation split” to “fit it while paying for complexity.”

### 3. Grid search is exhaustive over a chosen lattice

Grid search first chooses a finite lattice of candidate hyperparameters, evaluates every point, and returns the best measured validation score. Its strength is transparency: if the best point lies in the grid, it will be seen. Its weakness is also visible: a coarse grid can miss good values between the ticks, and multi-dimensional grids grow multiplicatively.

In [ ]:
depths_w = np.array([1, 2, 3, 4, 5, 6])  # candidate model depths.
regs_w = np.array([0.0, 0.1, 0.3, 1.0])  # candidate regularization strengths.
D_w, L_w = np.meshgrid(depths_w, regs_w, indexing="ij")  # every depth-reg pair.
score_grid_w = 0.20 + 0.015 * (D_w - 4) ** 2 + 0.08 * (np.log10(L_w + 0.08) + 0.7) ** 2
best_flat_w = int(np.argmin(score_grid_w))
best_pair_w = np.unravel_index(best_flat_w, score_grid_w.shape)
print("number of grid trials:", score_grid_w.size)
print("best depth, reg:", depths_w[best_pair_w[0]], regs_w[best_pair_w[1]])
print("best score:", round(float(score_grid_w[best_pair_w]), 3))
assert score_grid_w.size == 24
assert depths_w[best_pair_w[0]] == 4

▶ What you'll see: grid search evaluates all 24 combinations and finds the best candidate on the lattice.

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.imshow(score_grid_w, cmap="viridis_r", aspect="auto")
plt.colorbar(label="validation score (lower is better)")
plt.scatter([best_pair_w[1]], [best_pair_w[0]], color="red", s=80, label="best grid point")
plt.xticks(range(len(regs_w)), regs_w)
plt.yticks(range(len(depths_w)), depths_w)
plt.xlabel("regularization")
plt.ylabel("depth")
plt.title("3: exhaustive grid scores")
plt.legend()
plt.show()

▶ What you'll see: every lattice point is scored, and the red point marks the lowest cell.

*Why it's done this way:* a grid converts a continuous or large search space into a finite optimization problem. The Cartesian product makes the method simple and reproducible, but the trial count is `len(depths) × len(regs)`. That multiplication is why grid search becomes expensive when many knobs matter.

### 4. Random search spends trials on more unique values of important knobs

Random search samples candidate settings from distributions instead of committing to a fixed lattice. This is surprisingly powerful when only a few hyperparameters matter: the random trials explore many distinct values of the important knobs instead of wasting every combination of unimportant knobs.

In [ ]:
rng_w = np.random.default_rng(0)
n_trials_w = 24
random_depth_w = rng_w.integers(1, 8, size=n_trials_w)  # sampled integer depth.
random_reg_w = 10 ** rng_w.uniform(-3, 0, size=n_trials_w)  # sampled log-uniform regularization.
random_scores_w = 0.18 + 0.018 * (random_depth_w - 5) ** 2 + 0.06 * (np.log10(random_reg_w) + 1.0) ** 2
best_random_w = int(np.argmin(random_scores_w))
print("best random depth:", int(random_depth_w[best_random_w]))
print("best random reg:", round(float(random_reg_w[best_random_w]), 4))
print("best random score:", round(float(random_scores_w[best_random_w]), 3))
assert n_trials_w == 24
assert random_scores_w[best_random_w] == np.min(random_scores_w)

▶ What you'll see: the best sampled setting is chosen from the same number of trials as the grid example.

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(random_depth_w, random_reg_w, c=random_scores_w, cmap="viridis_r", s=70)
plt.scatter([random_depth_w[best_random_w]], [random_reg_w[best_random_w]], color="red", s=120, marker="*")
plt.yscale("log")
plt.colorbar(label="validation score")
plt.title("4: random search samples the space")
plt.xlabel("depth")
plt.ylabel("regularization")
plt.show()

▶ What you'll see: points are scattered through the continuous range, and the star marks the best sampled trial.

*Why it's done this way:* random sampling avoids tying every trial to a predetermined lattice. Sampling regularization on a log scale is especially important because values like `0.001`, `0.01`, and `0.1` are equally meaningful multiplicative changes. Random search is still blind, but it often covers useful ranges better than a coarse grid with the same budget.

### 5. Bayesian-style search uses previous trials to choose promising next points

Bayesian optimization is a family of methods, but the core idea is simple: fit a cheap **surrogate** to the observed trial results, then evaluate new settings where the surrogate predicts strong performance. Here we build a tiny surrogate from radial weights around prior trials. It is not a production Bayesian optimizer, but it shows the essential loop: observe, model, choose.

In [ ]:
x_trials_w = np.array([0.0, 0.7, 1.4, 2.4, 3.2])  # tried hyperparameter values on a 1-D scale.
y_trials_w = np.array([0.62, 0.34, 0.25, 0.31, 0.55])  # observed validation losses.
x_grid_w = np.linspace(0, 3.2, 100)
bandwidth_w = 0.45
weights_w = np.exp(-0.5 * ((x_grid_w[:, None] - x_trials_w[None, :]) / bandwidth_w) ** 2)
surrogate_w = (weights_w @ y_trials_w) / weights_w.sum(axis=1)
next_x_w = float(x_grid_w[int(np.argmin(surrogate_w))])
print("surrogate suggests next x:", round(next_x_w, 3))
print("best observed x:", x_trials_w[int(np.argmin(y_trials_w))])
assert 0.9 < next_x_w < 1.9

▶ What you'll see: the surrogate recommends searching near the already-good middle region.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(x_grid_w, surrogate_w, color="teal", label="surrogate mean")
plt.scatter(x_trials_w, y_trials_w, color="black", label="observed trials")
plt.axvline(next_x_w, color="crimson", linestyle="--", label="next trial")
plt.title("5: surrogate-guided next setting")
plt.xlabel("hyperparameter value")
plt.ylabel("validation loss")
plt.legend()
plt.show()

▶ What you'll see: the next trial is placed where nearby observed scores make the surrogate lowest.

*Why it's done this way:* a surrogate makes search adaptive. Instead of treating each trial as isolated, it assumes nearby hyperparameter settings tend to have related validation scores. The optimizer then spends trials near promising regions while still needing exploration rules in real systems to avoid being trapped by early noise.

### 6. Hyperband allocates more budget only to settings that earn it

Some settings look bad quickly; others deserve more training epochs, data, or compute. Hyperband-style methods exploit this by testing many settings cheaply, keeping a strong fraction, and increasing the resource budget for survivors. The math is a repeated selection rule: evaluate at a small budget, rank, keep, and allocate more.

In [ ]:
configs_w = np.arange(8)
small_budget_w = np.array([0.44, 0.39, 0.52, 0.31, 0.47, 0.29, 0.41, 0.36])
keep_w = 4
survivors_w = np.argsort(small_budget_w)[:keep_w]
print("small-budget survivors:", survivors_w)
assert np.array_equal(survivors_w, np.array([5, 3, 7, 1]))

▶ What you'll see: only the four lowest small-budget losses survive to the next round.

In [ ]:
large_budget_w = np.array([0.27, 0.25, 0.33, 0.30])  # losses for survivors [5,3,7,1] after more resources.
winner_pos_w = int(np.argmin(large_budget_w))
winner_config_w = int(survivors_w[winner_pos_w])
print("large-budget scores for survivors:", large_budget_w)
print("winner config:", winner_config_w)
assert winner_config_w == 3
assert round(float(np.min(large_budget_w)), 2) == 0.25

▶ What you'll see: the best final setting is selected only after a larger-budget comparison among survivors.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(configs_w, small_budget_w, color="lightgray", label="small budget")
plt.bar(survivors_w, small_budget_w[survivors_w], color="steelblue", label="promoted")
plt.scatter(survivors_w, large_budget_w, color="crimson", s=80, label="large-budget score")
plt.title("6: successive allocation of resources")
plt.xlabel("configuration id")
plt.ylabel("validation loss")
plt.legend()
plt.show()

▶ What you'll see: many gray candidates are stopped early; only promoted candidates receive the expensive second measurement.

*Why it's done this way:* compute is itself a hyperparameter-search resource. Hyperband reduces wasted budget by treating early performance as a screening signal. The risk is that a slow-starting configuration might be eliminated, so the budget schedule must match the learning dynamics of the models being tuned.

### 7. Validation gaps must be read against noise and stability

Hyperparameter search can overfit the validation set because many trials create many chances for a lucky score. A gap of `0.052` sounds concrete, but it is evidence only if it is large relative to resampling noise. Stability checks repeat the comparison across splits or bootstrap samples and ask whether the same setting keeps winning.

In [ ]:
baseline_score_w = 0.350
flexible_score_w = 0.402
gap_w = flexible_score_w - baseline_score_w
relative_gap_w = gap_w / flexible_score_w
stable_score_w = 0.80 * baseline_score_w
print("gap:", round(gap_w, 3))
print("relative gap:", round(relative_gap_w, 3))
print("stabilized score:", round(stable_score_w, 3))
assert round(gap_w, 3) == 0.052
assert round(relative_gap_w, 3) == 0.129
assert round(stable_score_w, 3) == 0.280

▶ What you'll see: the stabilized setting improves the decision score from `0.350` to `0.280`.

In [ ]:
scores_w = np.array([baseline_score_w, flexible_score_w, stable_score_w])
labels_w = ["baseline", "flexible", "stabilized"]
winner_w = int(np.argmin(scores_w))
print("winner:", labels_w[winner_w])
print("winning score:", round(float(scores_w[winner_w]), 3))
assert labels_w[winner_w] == "stabilized"
assert round(float(np.min(scores_w)), 3) == 0.280
plt.figure(figsize=(5, 3))
plt.bar(labels_w, scores_w, color=["steelblue", "orange", "seagreen"])
plt.title("7: compare full decision scores")
plt.ylabel("score (lower is better)")
plt.show()

▶ What you'll see: the stabilized score is the lowest, so it is the one to carry forward in the toy decision.

*Why it's done this way:* the final choice should be based on the full score and on whether the improvement is stable enough to trust. A small validation edge can be sampling luck, especially after many trials. Repeating comparisons or preferring simpler settings when gaps are tiny keeps search from turning into validation-set memorization.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, synthetic losses, random sampling, and numerical assertions.
import matplotlib.pyplot as plt # load Matplotlib for all score curves, heatmaps, scatter plots, and budget charts.
np.random.seed(0) # make examples reproducible when they use NumPy's global random state.

def validation_curve(x, center=0.35, width=0.16, floor=0.20): # define a smooth toy validation objective.
    return floor + width * (x - center) ** 2 # lower values near center mimic a good hyperparameter region.

def choose_best(scores): # define the basic argmin decision used throughout the lesson.
    scores = np.asarray(scores, dtype=float) # convert inputs to a numeric array for safe indexing.
    return int(np.argmin(scores)), float(np.min(scores)) # return both the winning index and the winning score.

def plot_scores(xs, scores, title): # define a compact helper for one-dimensional search traces.
    plt.figure(figsize=(5, 3)) # create a notebook-friendly figure.
    plt.plot(xs, scores, marker="o", color="teal") # draw settings against their validation scores.
    plt.title(title) # label the figure with the current example.
    plt.xlabel("hyperparameter setting") # label the x-axis generically.
    plt.ylabel("validation score") # lower validation score is better in this lesson.
    plt.show() # display the figure.

▶ What you'll see: the helper functions used by the worked examples are defined once.

## 🟢 Basics (warm-up)

### Basic 1 — Average per-example validation losses

**Goal.** Compute the empirical validation risk from individual losses, because every search method ranks settings by an averaged performance number. We build it in 2 steps.

In [ ]:
losses_b1 = np.array([0.268, 0.135, 0.437]) # store the three verified per-example losses from the lesson block.
print("losses:", losses_b1) # inspect the raw pieces before averaging.
assert np.allclose(losses_b1, np.array([0.268, 0.135, 0.437])) # verify the lesson inputs.

▶ What you'll see: the three loss values that will be averaged into one validation risk.

In [ ]:
risk_b1 = float(np.mean(losses_b1)) # average the losses to get empirical validation risk.
print("validation risk:", round(risk_b1, 3)) # inspect the averaged score.
assert round(risk_b1, 3) == 0.280 # verify (0.268+0.135+0.437)/3.
plt.figure(figsize=(4, 3)) # create a small bar chart.
plt.bar(["ex1", "ex2", "ex3", "mean"], list(losses_b1) + [risk_b1], color=["gray", "gray", "gray", "teal"]) # compare examples to their average.
plt.title("Basic 1: validation risk is an average") # title the figure.
plt.ylabel("loss") # label the loss scale.
plt.show() # display the plot.

▶ What you'll see: the mean bar lands at `0.280`, summarizing the three per-example losses.

👀 Takeaway: hyperparameter search starts by turning per-example losses into one comparable validation score.

### Basic 2 — Add a cost term to the score

**Goal.** Add complexity or operational cost to raw risk, because the selection criterion may penalize settings that are expensive or too flexible. We build it in 2 steps.

In [ ]:
risk_b2 = 0.280 # use the verified average validation risk.
cost_b2 = 0.070 # use the verified cost term from the lesson block.
print("risk:", risk_b2, "cost:", cost_b2) # inspect both terms before combining.
assert round(risk_b2 + cost_b2, 3) == 0.350 # verify the score arithmetic.

▶ What you'll see: the raw risk and the extra cost term are separate ingredients.

In [ ]:
score_b2 = risk_b2 + cost_b2 # compute the full decision score.
print("decision score:", round(score_b2, 3)) # inspect the score used for ranking.
assert round(score_b2, 3) == 0.350 # verify the lesson score.
plt.figure(figsize=(4, 3)) # create a compact decomposition plot.
plt.bar(["risk", "cost", "score"], [risk_b2, cost_b2, score_b2], color=["steelblue", "orange", "seagreen"]) # display the additive score parts.
plt.title("Basic 2: score includes cost") # title the figure.
plt.ylabel("value") # label the numeric scale.
plt.show() # display the plot.

▶ What you'll see: the score bar equals the risk bar plus the cost bar.

👀 Takeaway: the model-selection target is the full score, not whichever component looks nicest alone.

### Basic 3 — Choose the lower of two validation scores

**Goal.** Compare two candidate settings with `argmin`, because hyperparameter search is repeated score comparison. We build it in 2 steps.

In [ ]:
names_b3 = np.array(["baseline", "flexible"]) # name two candidate settings.
scores_b3 = np.array([0.350, 0.402]) # store their full decision scores.
print("candidate scores:", dict(zip(names_b3, scores_b3))) # inspect the comparison table.
assert round(float(scores_b3[1] - scores_b3[0]), 3) == 0.052 # verify the lesson gap.

▶ What you'll see: the flexible alternative is worse here because its score is higher.

In [ ]:
winner_b3, best_score_b3 = choose_best(scores_b3) # choose the index and score with the helper.
print("winner:", names_b3[winner_b3], "score:", best_score_b3) # inspect the selected setting.
assert names_b3[winner_b3] == "baseline" # lower score wins.
plt.figure(figsize=(4, 3)) # create a simple comparison plot.
plt.bar(names_b3, scores_b3, color=["teal", "gray"]) # show both candidate scores.
plt.title("Basic 3: lower score wins") # title the figure.
plt.ylabel("decision score") # label score scale.
plt.show() # display the plot.

▶ What you'll see: the baseline bar is lower, so it is selected.

👀 Takeaway: the whole search loop is built from consistent, lower-is-better comparisons.

### Basic 4 — Compute an absolute and relative gap

**Goal.** Measure the size of a win, because a small gap may be less persuasive than a stable large gap. We build it in 2 steps.

In [ ]:
baseline_b4 = 0.350 # score for the simpler setting.
flexible_b4 = 0.402 # score for the tempting alternative.
gap_b4 = flexible_b4 - baseline_b4 # absolute score difference.
print("absolute gap:", round(gap_b4, 3)) # inspect how much worse the alternative is.
assert round(gap_b4, 3) == 0.052 # verify the lesson gap.

▶ What you'll see: the absolute gap is `0.052`.

In [ ]:
relative_gap_b4 = gap_b4 / flexible_b4 # scale the gap by the larger alternative score.
print("relative gap:", round(relative_gap_b4, 3)) # inspect the percentage-like improvement.
assert round(relative_gap_b4, 3) == 0.129 # verify the lesson relative gap.
plt.figure(figsize=(4, 3)) # create a gap visualization.
plt.bar(["absolute", "relative"], [gap_b4, relative_gap_b4], color=["steelblue", "purple"]) # compare two gap readings.
plt.title("Basic 4: gap size matters") # title the figure.
plt.ylabel("gap") # label the gap scale.
plt.show() # display the plot.

▶ What you'll see: the relative gap is about `12.9%` of the flexible score.

👀 Takeaway: a score difference is evidence only after you understand its scale.

### Basic 5 — Apply a stabilizing multiplier

**Goal.** Compute the stabilized score from a 20% reduction, because some knobs deliberately trade flexibility for more reliable future performance. We build it in 2 steps.

In [ ]:
score_b5 = 0.350 # start from the baseline full decision score.
stability_multiplier_b5 = 0.80 # a 20% reduction means keeping 80% of the score.
print("multiplier:", stability_multiplier_b5) # inspect the stabilizing factor.
assert stability_multiplier_b5 == 0.80 # verify the intended reduction.

▶ What you'll see: the stabilizing knob is represented as a multiplier of `0.80`.

In [ ]:
stable_b5 = stability_multiplier_b5 * score_b5 # compute the stabilized score.
print("stabilized score:", round(stable_b5, 3)) # inspect the new decision value.
assert round(stable_b5, 3) == 0.280 # verify the lesson value.
plt.figure(figsize=(4, 3)) # create a before-after chart.
plt.bar(["before", "after"], [score_b5, stable_b5], color=["gray", "seagreen"]) # compare original and stabilized scores.
plt.title("Basic 5: stabilization lowers score") # title the figure.
plt.ylabel("decision score") # label the score scale.
plt.show() # display the plot.

▶ What you'll see: the stabilized bar drops from `0.350` to `0.280`.

👀 Takeaway: stability adjustments should be explicit arithmetic, not vague preference.

### Basic 6 — Select the minimum among three settings

**Goal.** Make the final toy decision among baseline, flexible, and stabilized settings, because real search often compares more than two candidates. We build it in 2 steps.

In [ ]:
labels_b6 = np.array(["baseline", "flexible", "stabilized"]) # name the three settings.
scores_b6 = np.array([0.350, 0.402, 0.280]) # store the final decision scores.
print("scores:", dict(zip(labels_b6, scores_b6))) # inspect all candidates before selecting.
assert round(float(np.min(scores_b6)), 3) == 0.280 # verify the lesson minimum.

▶ What you'll see: the stabilized candidate has the smallest score.

In [ ]:
winner_b6, best_b6 = choose_best(scores_b6) # select the lowest score.
print("chosen setting:", labels_b6[winner_b6], "score:", round(best_b6, 3)) # inspect the final choice.
assert labels_b6[winner_b6] == "stabilized" # verify the toy decision.
plt.figure(figsize=(5, 3)) # create a final comparison plot.
plt.bar(labels_b6, scores_b6, color=["steelblue", "orange", "seagreen"]) # show all candidates.
plt.title("Basic 6: final score comparison") # title the figure.
plt.ylabel("score") # label lower-is-better score scale.
plt.show() # display the plot.

▶ What you'll see: the stabilized score is the lowest bar and therefore the winner.

👀 Takeaway: carry forward the setting with the lowest full decision score.

### Basic 7 — Build a one-dimensional grid

**Goal.** Evaluate a fixed list of hyperparameter values, because grid search begins by deciding which candidates exist. We build it in 2 steps.

In [ ]:
xs_b7 = np.linspace(0.0, 1.0, 6) # create six evenly spaced candidate settings.
scores_b7 = validation_curve(xs_b7) # evaluate the toy validation curve at each setting.
print("grid settings:", np.round(xs_b7, 2)) # inspect the candidate list.
assert len(xs_b7) == 6 # verify the grid size.

▶ What you'll see: the grid contains six fixed values between 0 and 1.

In [ ]:
best_idx_b7, best_score_b7 = choose_best(scores_b7) # find the best grid point.
print("best grid x:", round(float(xs_b7[best_idx_b7]), 2), "score:", round(best_score_b7, 3)) # inspect the best candidate.
assert round(float(xs_b7[best_idx_b7]), 2) == 0.40 # verify the nearest grid point to the curve center.
plot_scores(xs_b7, scores_b7, "Basic 7: one-dimensional grid search") # visualize the search curve.

▶ What you'll see: the best grid point is near the curve's low region around `0.35`.

👀 Takeaway: grid search can only select among values you put on the grid.

### Basic 8 — Sample random hyperparameters

**Goal.** Draw candidate settings randomly, because random search explores continuous ranges without a fixed lattice. We build it in 2 steps.

In [ ]:
rng_b8 = np.random.default_rng(8) # use a local generator so the example is reproducible.
xs_b8 = rng_b8.uniform(0.0, 1.0, size=8) # sample eight random settings.
scores_b8 = validation_curve(xs_b8) # evaluate the same toy validation objective.
print("random settings:", np.round(xs_b8, 3)) # inspect the sampled candidates.
assert len(xs_b8) == 8 # verify the number of trials.

▶ What you'll see: the settings are irregularly spaced rather than arranged on a grid.

In [ ]:
best_idx_b8, best_score_b8 = choose_best(scores_b8) # choose the best sampled setting.
print("best random x:", round(float(xs_b8[best_idx_b8]), 3), "score:", round(best_score_b8, 3)) # inspect the best random trial.
assert scores_b8[best_idx_b8] == np.min(scores_b8) # verify argmin selection.
plt.figure(figsize=(5, 3)) # create a scatter plot of sampled scores.
plt.scatter(xs_b8, scores_b8, color="teal", s=70) # show each random trial.
plt.scatter([xs_b8[best_idx_b8]], [best_score_b8], color="red", s=110, marker="*") # highlight the winner.
plt.title("Basic 8: random search trials") # title the figure.
plt.xlabel("sampled setting") # label x-axis.
plt.ylabel("validation score") # label score axis.
plt.show() # display the plot.

▶ What you'll see: the winning random sample is the point closest to the low part of the curve.

👀 Takeaway: random search replaces a fixed lattice with sampled candidates from chosen distributions.

### Basic 9 — Count trials in a two-dimensional grid

**Goal.** Compute the Cartesian-product size, because grid-search cost multiplies across hyperparameters. We build it in 2 steps.

In [ ]:
depths_b9 = np.array([2, 4, 6, 8]) # candidate values for a first hyperparameter.
rates_b9 = np.array([0.001, 0.01, 0.1]) # candidate values for a second hyperparameter.
trial_count_b9 = len(depths_b9) * len(rates_b9) # Cartesian product count.
print("trial count:", trial_count_b9) # inspect how many model fits are required.
assert trial_count_b9 == 12 # verify 4 by 3 combinations.

▶ What you'll see: a small two-knob grid already needs 12 trials.

In [ ]:
D_b9, R_b9 = np.meshgrid(depths_b9, rates_b9, indexing="ij") # materialize all combinations.
print("grid shape:", D_b9.shape) # inspect the two-dimensional grid shape.
assert D_b9.size == trial_count_b9 # verify the materialized grid has the expected number of points.
plt.figure(figsize=(4.5, 3)) # create a grid-coordinate plot.
plt.scatter(D_b9.ravel(), R_b9.ravel(), color="purple") # show each combination as one point.
plt.yscale("log") # show rate values on their natural multiplicative scale.
plt.title("Basic 9: grid combinations") # title the figure.
plt.xlabel("depth") # label first hyperparameter.
plt.ylabel("learning rate") # label second hyperparameter.
plt.show() # display the plot.

▶ What you'll see: every depth is paired with every learning rate.

👀 Takeaway: grid search is easy to reason about, but its cost grows multiplicatively.

### Basic 10 — Sort candidates by score

**Goal.** Rank all tried settings, because hyperparameter search often reports the best few trials rather than only the winner. We build it in 2 steps.

In [ ]:
names_b10 = np.array(["trial0", "trial1", "trial2", "trial3", "trial4"]) # label five completed trials.
scores_b10 = np.array([0.42, 0.31, 0.28, 0.36, 0.33]) # store validation scores for the trials.
order_b10 = np.argsort(scores_b10) # sort from lowest to highest score.
print("sorted names:", names_b10[order_b10]) # inspect the ranking order.
assert names_b10[order_b10][0] == "trial2" # verify the best trial.

▶ What you'll see: `trial2` appears first because it has the lowest score.

In [ ]:
top3_b10 = names_b10[order_b10[:3]] # keep the three best trials.
print("top 3:", top3_b10) # inspect a short leaderboard.
assert len(top3_b10) == 3 # verify the requested leaderboard size.
plt.figure(figsize=(5, 3)) # create a leaderboard chart.
plt.bar(names_b10[order_b10], scores_b10[order_b10], color="seagreen") # plot scores in sorted order.
plt.title("Basic 10: ranked search results") # title the figure.
plt.ylabel("validation score") # label lower-is-better scores.
plt.show() # display the chart.

▶ What you'll see: bars rise from best to worst score in the sorted leaderboard.

👀 Takeaway: ranking trials makes it easier to inspect near-winners and not just the single argmin.

## 🟡 Easy

### Easy 1 — Implement grid search over regularization

**Goal.** Search a regularization grid and select the validation minimum, because this is the simplest complete hyperparameter-search loop. We build it in 3 steps.

In [ ]:
regs_e1 = np.array([0.0, 0.01, 0.1, 1.0, 10.0]) # candidate regularization strengths.
train_e1 = 0.08 + 0.05 * np.log10(regs_e1 + 1.01) # toy train loss rises gently with more penalty.
val_e1 = np.array([0.40, 0.31, 0.23, 0.27, 0.52]) # toy validation losses with a middle optimum.
print("regularization grid:", regs_e1) # inspect candidates.
assert len(regs_e1) == 5 # verify five candidates are evaluated.

▶ What you'll see: five regularization settings are ready to compare.

In [ ]:
best_idx_e1, best_score_e1 = choose_best(val_e1) # choose by validation loss.
best_reg_e1 = float(regs_e1[best_idx_e1]) # read the best regularization value.
print("best reg:", best_reg_e1, "best validation:", best_score_e1) # inspect the chosen setting.
assert best_reg_e1 == 0.1 # verify the intended optimum.

▶ What you'll see: `0.1` wins because it has the smallest validation score.

In [ ]:
plt.figure(figsize=(5, 3)) # create a grid-search curve.
plt.plot(regs_e1, train_e1, marker="o", label="train") # plot train scores.
plt.plot(regs_e1, val_e1, marker="o", label="validation") # plot validation scores.
plt.xscale("symlog") # show zero and log-spaced positive values readably.
plt.axvline(best_reg_e1, color="crimson", linestyle="--", label="chosen") # mark the selected regularization.
plt.title("Easy 1: regularization grid search") # title the figure.
plt.xlabel("regularization") # label x-axis.
plt.ylabel("loss") # label y-axis.
plt.legend() # show curve labels.
plt.show() # display the plot.

▶ What you'll see: validation loss has a U-shape, so the search chooses the middle rather than the most flexible setting.

👀 Takeaway: grid search is just evaluate all candidates, then choose the lowest validation score.

### Easy 2 — Compare grid search and random search with the same budget

**Goal.** Use the same trial budget for grid and random search, because search methods should be compared under equal cost. We build it in 3 steps.

In [ ]:
grid_x_e2 = np.linspace(0, 1, 10) # ten evenly spaced grid trials.
grid_scores_e2 = validation_curve(grid_x_e2, center=0.37, width=0.22, floor=0.18) # evaluate the toy objective.
rng_e2 = np.random.default_rng(2) # reproducible random search.
rand_x_e2 = rng_e2.uniform(0, 1, size=10) # ten random trials.
rand_scores_e2 = validation_curve(rand_x_e2, center=0.37, width=0.22, floor=0.18) # evaluate the same objective.
print("trial budgets:", len(grid_x_e2), len(rand_x_e2)) # confirm equal budgets.
assert len(grid_x_e2) == len(rand_x_e2) == 10 # verify fair comparison.

▶ What you'll see: both methods get exactly ten trials.

In [ ]:
grid_best_idx_e2, grid_best_e2 = choose_best(grid_scores_e2) # best grid score.
rand_best_idx_e2, rand_best_e2 = choose_best(rand_scores_e2) # best random score.
print("grid best:", round(grid_best_e2, 4), "random best:", round(rand_best_e2, 4)) # inspect the best score from each method.
assert grid_best_e2 <= np.max(grid_scores_e2) # sanity-check score range.
assert rand_best_e2 <= np.max(rand_scores_e2) # sanity-check score range.

▶ What you'll see: either method can win on a particular run, but the comparison uses equal cost.

In [ ]:
plt.figure(figsize=(5, 3)) # create a method comparison plot.
plt.scatter(grid_x_e2, grid_scores_e2, label="grid", color="steelblue") # show grid trials.
plt.scatter(rand_x_e2, rand_scores_e2, label="random", color="orange") # show random trials.
plt.scatter([grid_x_e2[grid_best_idx_e2], rand_x_e2[rand_best_idx_e2]], [grid_best_e2, rand_best_e2], color="red", marker="*", s=110) # highlight winners.
plt.title("Easy 2: same budget, different coverage") # title the plot.
plt.xlabel("setting") # label hyperparameter axis.
plt.ylabel("validation score") # label score axis.
plt.legend() # show method labels.
plt.show() # display the plot.

▶ What you'll see: grid points are evenly spaced, while random points can land closer or farther from the optimum.

👀 Takeaway: compare search strategies by best validation score at the same trial budget.

### Easy 3 — Use log-uniform sampling for scale-sensitive knobs

**Goal.** Sample learning rates on a log scale, because multiplicative changes matter more than additive changes for many optimization knobs. We build it in 3 steps.

In [ ]:
rng_e3 = np.random.default_rng(3) # reproducible random generator.
linear_rates_e3 = rng_e3.uniform(1e-4, 1.0, size=200) # sample learning rates uniformly on the raw scale.
log_rates_e3 = 10 ** rng_e3.uniform(-4, 0, size=200) # sample learning rates uniformly over powers of ten.
print("linear min/max:", round(float(linear_rates_e3.min()), 5), round(float(linear_rates_e3.max()), 3)) # inspect linear range.
print("log min/max:", round(float(log_rates_e3.min()), 5), round(float(log_rates_e3.max()), 3)) # inspect log range.
assert len(log_rates_e3) == 200 # verify sample count.

▶ What you'll see: both methods cover the same endpoints, but they distribute samples very differently.

In [ ]:
small_linear_e3 = int(np.sum(linear_rates_e3 < 0.01)) # count tiny learning rates under raw-uniform sampling.
small_log_e3 = int(np.sum(log_rates_e3 < 0.01)) # count tiny learning rates under log-uniform sampling.
print("samples below 0.01 — linear:", small_linear_e3, "log:", small_log_e3) # inspect coverage of small scales.
assert small_log_e3 > small_linear_e3 # log-uniform should allocate more samples to small orders of magnitude.

▶ What you'll see: log-uniform sampling spends many more trials below `0.01`.

In [ ]:
plt.figure(figsize=(5, 3)) # create a histogram comparison.
plt.hist(np.log10(linear_rates_e3), bins=20, alpha=0.7, label="raw uniform") # plot raw-uniform samples in log coordinates.
plt.hist(np.log10(log_rates_e3), bins=20, alpha=0.7, label="log uniform") # plot log-uniform samples in log coordinates.
plt.title("Easy 3: sampling learning rates by scale") # title the plot.
plt.xlabel("log10(learning rate)") # label log scale.
plt.ylabel("count") # label histogram counts.
plt.legend() # show sample method labels.
plt.show() # display the histogram.

▶ What you'll see: log-uniform sampling is much flatter across powers of ten.

👀 Takeaway: search distributions should match the scale on which the hyperparameter changes model behavior.

### Easy 4 — Run a tiny surrogate-guided search

**Goal.** Choose a next trial from a surrogate curve, because adaptive search uses previous results to guide future evaluations. We build it in 3 steps.

In [ ]:
x_obs_e4 = np.array([0.05, 0.25, 0.55, 0.90]) # hyperparameter values already tried.
y_obs_e4 = validation_curve(x_obs_e4, center=0.42, width=0.25, floor=0.19) + np.array([0.02, -0.01, 0.01, 0.03]) # observed noisy scores.
x_candidates_e4 = np.linspace(0, 1, 80) # candidate locations for the next trial.
print("observed trials:", len(x_obs_e4)) # inspect how much data the surrogate has.
assert len(x_obs_e4) == 4 # verify observation count.

▶ What you'll see: the adaptive method starts with four completed trials.

In [ ]:
width_e4 = 0.18 # radial smoothing width.
w_e4 = np.exp(-0.5 * ((x_candidates_e4[:, None] - x_obs_e4[None, :]) / width_e4) ** 2) # compute similarity from candidates to observations.
surrogate_e4 = (w_e4 @ y_obs_e4) / w_e4.sum(axis=1) # weighted average predicted score.
next_idx_e4, next_score_e4 = choose_best(surrogate_e4) # choose the candidate with lowest surrogate score.
next_x_e4 = float(x_candidates_e4[next_idx_e4]) # read next hyperparameter value.
print("next trial x:", round(next_x_e4, 3), "surrogate score:", round(next_score_e4, 3)) # inspect the recommendation.
assert 0.2 < next_x_e4 < 0.7 # verify it stays near promising observations.

▶ What you'll see: the surrogate recommends another trial near the promising middle region.

In [ ]:
plt.figure(figsize=(5, 3)) # create a surrogate plot.
plt.plot(x_candidates_e4, surrogate_e4, color="teal", label="surrogate") # show predicted score curve.
plt.scatter(x_obs_e4, y_obs_e4, color="black", label="observed") # show completed trials.
plt.axvline(next_x_e4, color="crimson", linestyle="--", label="next") # mark next candidate.
plt.title("Easy 4: surrogate-guided choice") # title the plot.
plt.xlabel("setting") # label x-axis.
plt.ylabel("score") # label y-axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: the next trial is placed where the surrogate curve is lowest.

👀 Takeaway: Bayesian-style search turns past trials into a model that proposes future trials.

### Easy 5 — Successive halving with two budgets

**Goal.** Promote only the best early configurations to a larger budget, because Hyperband-style search saves compute by stopping weak trials early. We build it in 3 steps.

In [ ]:
configs_e5 = np.arange(6) # six candidate configurations.
small_scores_e5 = np.array([0.48, 0.35, 0.44, 0.31, 0.52, 0.38]) # early validation scores after a small budget.
promote_e5 = np.argsort(small_scores_e5)[:3] # keep the three lowest early losses.
print("promoted configs:", promote_e5) # inspect which candidates survive.
assert np.array_equal(promote_e5, np.array([3, 1, 5])) # verify the early-ranking decision.

▶ What you'll see: configurations 3, 1, and 5 earn more resources.

In [ ]:
large_scores_e5 = np.array([0.28, 0.30, 0.27]) # final scores for promoted configs in the same order [3,1,5].
winner_pos_e5 = int(np.argmin(large_scores_e5)) # choose among promoted configurations.
winner_e5 = int(promote_e5[winner_pos_e5]) # map back to original config id.
print("winner after larger budget:", winner_e5) # inspect the final selection.
assert winner_e5 == 5 # verify the final winner.

▶ What you'll see: a candidate that was not first early can win after a larger budget.

In [ ]:
plt.figure(figsize=(5, 3)) # create a successive-halving plot.
plt.bar(configs_e5, small_scores_e5, color="lightgray", label="small budget") # show early results.
plt.bar(promote_e5, small_scores_e5[promote_e5], color="steelblue", label="promoted") # highlight survivors.
plt.scatter(promote_e5, large_scores_e5, color="crimson", s=80, label="large budget") # show later scores for survivors.
plt.title("Easy 5: promote then re-evaluate") # title the plot.
plt.xlabel("configuration") # label x-axis.
plt.ylabel("score") # label score scale.
plt.legend() # show legend.
plt.show() # display the plot.

▶ What you'll see: weak early candidates are discarded, while promoted candidates receive final measurements.

👀 Takeaway: resource-aware search spends expensive evaluations only on candidates that survive cheap screening.

## 🔴 Advanced

### Advanced 1 — Detect validation overfitting across many trials

**Goal.** Compare train and validation winners across many noisy trials, because the best validation score can be partly lucky when the search space is large. We build it in 4 steps.

In [ ]:
rng_a1 = np.random.default_rng(11) # reproducible simulation.
n_trials_a1 = 80 # number of hyperparameter trials.
complexity_a1 = np.linspace(0, 1, n_trials_a1) # increasing model flexibility.
true_risk_a1 = 0.24 + 0.18 * (complexity_a1 - 0.45) ** 2 # underlying generalization curve.
train_a1 = true_risk_a1 - 0.10 * complexity_a1 + rng_a1.normal(0, 0.01, n_trials_a1) # training favors complexity.
val_a1 = true_risk_a1 + rng_a1.normal(0, 0.025, n_trials_a1) # validation is noisy estimate of true risk.
print("trials:", n_trials_a1) # inspect search size.
assert n_trials_a1 == 80 # verify trial count.

▶ What you'll see: the simulation contains many chances for a lucky validation score.

In [ ]:
train_winner_a1 = int(np.argmin(train_a1)) # trial with lowest training loss.
val_winner_a1 = int(np.argmin(val_a1)) # trial with lowest validation loss.
true_winner_a1 = int(np.argmin(true_risk_a1)) # trial with lowest underlying risk.
print("train winner:", train_winner_a1, "validation winner:", val_winner_a1, "true winner:", true_winner_a1) # inspect mismatch.
assert train_winner_a1 > true_winner_a1 # training winner is more complex in this setup.

▶ What you'll see: the training winner is shifted toward high flexibility.

In [ ]:
optimism_a1 = float(true_risk_a1[val_winner_a1] - val_a1[val_winner_a1]) # how lucky the selected validation score was.
print("validation optimism at selected trial:", round(optimism_a1, 3)) # inspect selected-trial noise.
assert optimism_a1 > 0 # selected validation score is below its true risk in this run.

▶ What you'll see: the selected validation result is optimistically low relative to its true risk.

In [ ]:
plt.figure(figsize=(5, 3)) # create an overfitting diagnostic plot.
plt.plot(complexity_a1, true_risk_a1, label="true risk", color="black") # show underlying curve.
plt.scatter(complexity_a1, val_a1, s=20, alpha=0.7, label="validation trials", color="teal") # show noisy validation measurements.
plt.axvline(complexity_a1[val_winner_a1], color="crimson", linestyle="--", label="selected") # mark selected trial.
plt.title("Advanced 1: many trials invite lucky validation wins") # title plot.
plt.xlabel("complexity") # label complexity axis.
plt.ylabel("risk") # label risk axis.
plt.legend() # show labels.
plt.show() # display plot.

▶ What you'll see: among many noisy points, the selected minimum can lie below the true curve.

👀 Takeaway: more trials make search stronger, but they also increase the need for stable validation protocols.

### Advanced 2 — Nested validation prevents test-set leakage

**Goal.** Separate tuning from final evaluation, because using test data to choose hyperparameters leaks information into the reported result. We build it in 4 steps.

In [ ]:
rng_a2 = np.random.default_rng(12) # reproducible noisy scores.
settings_a2 = np.arange(12) # twelve hyperparameter settings.
true_test_risk_a2 = 0.22 + 0.012 * (settings_a2 - 5) ** 2 # fixed underlying test risk curve.
validation_a2 = true_test_risk_a2 + rng_a2.normal(0, 0.025, size=settings_a2.size) # validation estimates for tuning.
test_a2 = true_test_risk_a2 + rng_a2.normal(0, 0.018, size=settings_a2.size) # separate test estimates for reporting.
print("settings:", settings_a2) # inspect settings.
assert len(settings_a2) == 12 # verify candidate count.

▶ What you'll see: validation and test are two different noisy views of the same underlying risk.

In [ ]:
chosen_by_val_a2 = int(np.argmin(validation_a2)) # correct tuning choice.
chosen_by_test_a2 = int(np.argmin(test_a2)) # leaky choice that peeks at the test set.
print("chosen by validation:", chosen_by_val_a2) # inspect correct selection.
print("chosen by test (leaky):", chosen_by_test_a2) # inspect leakage selection.
assert test_a2[chosen_by_test_a2] <= test_a2[chosen_by_val_a2] # peeking cannot report a worse test minimum.

▶ What you'll see: picking by test can look better because it optimizes the report itself.

In [ ]:
reported_nested_a2 = float(test_a2[chosen_by_val_a2]) # honest final test score after validation tuning.
reported_leaky_a2 = float(test_a2[chosen_by_test_a2]) # optimistic score from test-set tuning.
print("nested report:", round(reported_nested_a2, 3), "leaky report:", round(reported_leaky_a2, 3)) # compare reports.
assert reported_leaky_a2 <= reported_nested_a2 # verify leakage optimism.

▶ What you'll see: the leaky report is at least as flattering as the honest nested report.

In [ ]:
plt.figure(figsize=(5, 3)) # create a nested-validation plot.
plt.plot(settings_a2, validation_a2, marker="o", label="validation for tuning") # show validation curve.
plt.plot(settings_a2, test_a2, marker="o", label="test for final report") # show test curve.
plt.axvline(chosen_by_val_a2, color="teal", linestyle="--", label="validation choice") # mark honest choice.
plt.axvline(chosen_by_test_a2, color="crimson", linestyle=":", label="test-peek choice") # mark leaky choice.
plt.title("Advanced 2: tune on validation, report on test") # title plot.
plt.xlabel("setting id") # label setting axis.
plt.ylabel("score") # label score axis.
plt.legend() # show labels.
plt.show() # display plot.

▶ What you'll see: the tuning split decides the setting, while the test split only measures the chosen setting once.

👀 Takeaway: never let the final test score participate in hyperparameter selection.

### Advanced 3 — Combine score and uncertainty with repeated splits

**Goal.** Repeat validation over several splits and compare confidence intervals, because a tiny mean win can disappear under sampling noise. We build it in 4 steps.

In [ ]:
rng_a3 = np.random.default_rng(13) # reproducible repeated-split simulation.
repeats_a3 = 20 # number of validation repeats.
score_A_a3 = rng_a3.normal(0.300, 0.030, size=repeats_a3) # repeated scores for setting A.
score_B_a3 = rng_a3.normal(0.286, 0.035, size=repeats_a3) # repeated scores for setting B.
print("repeat count:", repeats_a3) # inspect number of split estimates.
assert len(score_A_a3) == len(score_B_a3) == 20 # verify paired repeat count.

▶ What you'll see: each setting has a distribution of validation scores, not just one number.

In [ ]:
mean_A_a3 = float(np.mean(score_A_a3)) # mean score for setting A.
mean_B_a3 = float(np.mean(score_B_a3)) # mean score for setting B.
se_diff_a3 = float(np.std(score_A_a3 - score_B_a3, ddof=1) / np.sqrt(repeats_a3)) # standard error of paired differences.
mean_diff_a3 = mean_A_a3 - mean_B_a3 # positive means B is better because lower score.
print("mean A:", round(mean_A_a3, 3), "mean B:", round(mean_B_a3, 3)) # inspect average scores.
print("mean gap A-B:", round(mean_diff_a3, 3), "SE:", round(se_diff_a3, 3)) # inspect gap versus noise.
assert se_diff_a3 > 0 # verify uncertainty is nonzero.

▶ What you'll see: the mean gap must be interpreted relative to its standard error.

In [ ]:
ci_low_a3 = mean_diff_a3 - 1.96 * se_diff_a3 # lower approximate 95% interval for paired gap.
ci_high_a3 = mean_diff_a3 + 1.96 * se_diff_a3 # upper approximate 95% interval for paired gap.
stable_win_a3 = ci_low_a3 > 0 # B stably beats A only if the whole gap interval is positive.
print("95% gap interval:", round(ci_low_a3, 3), round(ci_high_a3, 3)) # inspect uncertainty interval.
print("stable B win:", stable_win_a3) # inspect decision stability.
assert isinstance(stable_win_a3, (bool, np.bool_)) # verify boolean decision.

▶ What you'll see: the stability decision depends on whether the interval clears zero.

In [ ]:
plt.figure(figsize=(5, 3)) # create a repeated-split comparison plot.
plt.boxplot([score_A_a3, score_B_a3], labels=["A", "B"]) # show score distributions.
plt.title("Advanced 3: repeated validation scores") # title plot.
plt.ylabel("validation score") # label score axis.
plt.show() # display plot.

▶ What you'll see: overlapping boxes warn that the apparent winner may be uncertain.

👀 Takeaway: when validation gaps are small, repeated splits help distinguish real improvement from noise.

### Advanced 4 — Multi-objective search with accuracy and latency

**Goal.** Add a latency penalty to validation loss, because production hyperparameter choices often trade predictive quality against operational cost. We build it in 4 steps.

In [ ]:
settings_a4 = np.arange(7) # seven candidate configurations.
val_loss_a4 = np.array([0.36, 0.30, 0.26, 0.24, 0.235, 0.233, 0.232]) # predictive loss improves slowly.
latency_ms_a4 = np.array([15, 22, 35, 55, 85, 130, 210]) # latency rises quickly with model size.
print("validation losses:", val_loss_a4) # inspect predictive quality.
print("latencies:", latency_ms_a4) # inspect operational cost.
assert val_loss_a4[-1] == np.min(val_loss_a4) # verify the largest model has best raw loss.

▶ What you'll see: raw validation loss favors the slowest configuration.

In [ ]:
latency_penalty_a4 = 0.0008 * latency_ms_a4 # convert milliseconds into score units.
full_score_a4 = val_loss_a4 + latency_penalty_a4 # combine predictive and operational cost.
raw_best_a4 = int(np.argmin(val_loss_a4)) # best by raw loss.
full_best_a4 = int(np.argmin(full_score_a4)) # best by full score.
print("raw best:", raw_best_a4, "full-score best:", full_best_a4) # inspect how penalty changes choice.
assert full_best_a4 < raw_best_a4 # verify latency penalty chooses a smaller setting.

▶ What you'll see: the best full score is not the slowest raw-loss winner.

In [ ]:
print("full scores:", np.round(full_score_a4, 3)) # inspect the combined objective.
assert np.all(full_score_a4 >= val_loss_a4) # verify adding latency never lowers score.

▶ What you'll see: every full score includes a nonnegative latency penalty.

In [ ]:
plt.figure(figsize=(5, 3)) # create a multi-objective comparison plot.
plt.plot(settings_a4, val_loss_a4, marker="o", label="validation loss") # plot raw predictive loss.
plt.plot(settings_a4, full_score_a4, marker="o", label="loss + latency penalty") # plot full objective.
plt.axvline(full_best_a4, color="crimson", linestyle="--", label="chosen") # mark chosen setting.
plt.title("Advanced 4: include operational cost") # title plot.
plt.xlabel("setting id") # label setting axis.
plt.ylabel("score") # label score axis.
plt.legend() # show labels.
plt.show() # display plot.

▶ What you'll see: latency shifts the selected configuration away from the largest model.

👀 Takeaway: hyperparameter search should optimize the score that matches the real deployment decision.

### Advanced 5 — Hyperband can miss slow starters

**Goal.** Show a failure mode of early stopping, because a configuration with weak early performance may become best after enough resource. We build it in 4 steps.

In [ ]:
configs_a5 = np.array(["fast-small", "steady", "slow-large", "noisy"]) # candidate names.
budgets_a5 = np.array([1, 3, 9]) # resource levels such as epochs or data fractions.
losses_a5 = np.array([[0.32, 0.30, 0.31], [0.40, 0.29, 0.25], [0.55, 0.33, 0.20], [0.36, 0.38, 0.34]]) # rows=configs, cols=budgets.
print("small-budget losses:", dict(zip(configs_a5, losses_a5[:, 0]))) # inspect early scores.
assert losses_a5.shape == (4, 3) # verify matrix layout.

▶ What you'll see: `slow-large` looks worst at the smallest budget.

In [ ]:
promoted_a5 = np.argsort(losses_a5[:, 0])[:2] # promote only two best at budget 1.
print("promoted after budget 1:", configs_a5[promoted_a5]) # inspect survivors.
assert "slow-large" not in configs_a5[promoted_a5] # verify slow starter is eliminated.

▶ What you'll see: the slow-starting configuration does not survive early screening.

In [ ]:
full_budget_winner_a5 = int(np.argmin(losses_a5[:, -1])) # best if all configs reached largest budget.
hyperband_winner_a5 = int(promoted_a5[np.argmin(losses_a5[promoted_a5, -1])]) # best among promoted configs.
print("best if fully trained:", configs_a5[full_budget_winner_a5]) # inspect true large-budget winner.
print("best among promoted:", configs_a5[hyperband_winner_a5]) # inspect early-stopping winner.
assert configs_a5[full_budget_winner_a5] == "slow-large" # verify the missed winner.

▶ What you'll see: the best fully trained configuration was eliminated early.

In [ ]:
plt.figure(figsize=(5, 3)) # create learning-curve comparison.
for i_a5, name_a5 in enumerate(configs_a5): # plot every configuration across budgets.
    plt.plot(budgets_a5, losses_a5[i_a5], marker="o", label=name_a5) # draw one resource curve.
plt.xscale("log") # budgets often grow multiplicatively.
plt.title("Advanced 5: early stopping can miss slow starters") # title plot.
plt.xlabel("resource budget") # label budget axis.
plt.ylabel("validation loss") # label loss axis.
plt.legend() # show config labels.
plt.show() # display plot.

▶ What you'll see: `slow-large` starts poorly but becomes best at the largest budget.

👀 Takeaway: Hyperband saves compute, but its early budget must be large enough to reveal promising slow-learning settings.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Hyperparameter search spends trials where good settings are likely, while guarding against lucky validation noise.

Search chooses settings outside ordinary fitting, so the validation protocol becomes part of the method. Grid and random search give reproducible baselines, while staged allocation imitates Hyperband by spending more budget on promising candidates.

Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np

from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import log_loss
from sklearn.metrics import recall_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsOneClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
np.random.seed(7)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def safe_split(X, y, test_size=0.4):
    counts = np.bincount(np.asarray(y))
    can_stratify = counts.min() >= 2
    stratify = y if can_stratify else None
    return train_test_split(X, y, test_size=test_size, random_state=0, stratify=stratify)


def scaled_train_test(X, y, test_size=0.4):
    x_tr, x_te, y_tr, y_te = safe_split(X, y, test_size=test_size)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def logistic_model(**kwargs):
    params = dict(max_iter=3000, solver="lbfgs")
    params.update(kwargs)
    return LogisticRegression(**params)


def predict_proba_or_scores(model, X, labels):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
    else:
        scores = model.decision_function(X)
        if scores.ndim == 1:
            scores = np.column_stack([-scores, scores])
        scores = scores - scores.max(axis=1, keepdims=True)
        proba = np.exp(scores)
        proba = proba / proba.sum(axis=1, keepdims=True)
    if proba.shape[1] == len(labels):
        return proba
    aligned = np.zeros((len(X), len(labels)))
    for j, label in enumerate(model.classes_):
        idx = list(labels).index(label)
        aligned[:, idx] = proba[:, j]
    aligned = np.clip(aligned, 1e-9, 1.0)
    aligned = aligned / aligned.sum(axis=1, keepdims=True)
    return aligned


def model_log_loss(model, x_te, y_te, labels):
    proba = predict_proba_or_scores(model, x_te, labels)
    return float(log_loss(y_te, proba, labels=labels))


def print_table(rows, headers):
    widths = [len(h) for h in headers]
    for row in rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))
    fmt = "  ".join("{:" + str(w) + "}" for w in widths)
    print(fmt.format(*headers))
    print(fmt.format(*["-" * w for w in widths]))
    for row in rows:
        print(fmt.format(*row))


def plot_summary(names, metrics, title, ylabel):
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    flat = axes.ravel()
    for idx, (name, X, y) in enumerate(clf_ladder()):
        ax = flat[idx]
        sample = X[:, :2]
        ax.scatter(sample[:, 0], sample[:, 1], c=y, cmap="viridis", s=20, alpha=0.8)
        ax.set_title(name.split("(")[0].strip())
        ax.set_xticks([])
        ax.set_yticks([])
    ax = flat[-1]
    ax.plot(range(1, len(metrics) + 1), metrics, marker="o")
    ax.set_title(title)
    ax.set_xlabel("rung")
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(1, len(metrics) + 1))
    fig.tight_layout()
    plt.show()


## The concept, built once on D1
The lesson formula is $$\lambda^*=\arg\min_{\lambda\in\Lambda} R_{val}(\hat f_\lambda)$$
We first reproduce the lesson arithmetic exactly, then reuse the corresponding real technique below.

In [ ]:

def hyperparameter_search_grid_random_bayesian_method(losses, cost, alternative):
    risk = float(np.mean(losses))
    score = risk + cost
    gap = alternative - score
    return risk, score, gap

losses = np.array([0.268, 0.135, 0.437])
risk, score, gap = hyperparameter_search_grid_random_bayesian_method(losses, 0.070, 0.402)
print(f"lesson risk={risk:.3f}, score={score:.3f}, gap={gap:.3f}")
assert round(risk, 3) == 0.280
assert round(score, 3) == 0.350
assert round(gap, 3) == 0.052


Now connect the arithmetic to a reusable model-selection habit: compute the validation metric, add any stated cost, and compare the gap to an alternative. The assert guards make the notebook reproducible.

In [ ]:
lesson_losses = np.array([0.268, 0.135, 0.437])
lesson_risk, lesson_score, lesson_gap = hyperparameter_search_grid_random_bayesian_method(lesson_losses, 0.070, 0.402)
print(round(lesson_risk, 3), round(lesson_score, 3), round(lesson_gap, 3))

## The dataset ladder
The same code runs from a hand-built D1 through real D4/D5 data. Each rung reports shape, classes, and a tiny sample so leakage, search, feature work, imbalance, and strategy choices stay inspectable.

In [ ]:

rows = []
for idx, (name, X, y) in enumerate(clf_ladder(), 1):
    classes, counts = np.unique(y, return_counts=True)
    class_summary = ", ".join([f"{cls}:{cnt}" for cls, cnt in zip(classes, counts)])
    rows.append([f"D{idx}", name, str(X.shape), class_summary, np.array2string(X[:2, :min(3, X.shape[1])], precision=2)])
print_table(rows, ["rung", "name", "shape", "classes", "sample"])


## Run the same method across D1–D5
One metric is collected per rung so the summary curve is comparable.

In [ ]:

def search_experiment(X, y):
    x_tr, x_te, y_tr, y_te = scaled_train_test(X, y)
    labels = np.unique(y)
    if len(y_tr) < 12:
        simple = SVC(C=1.0, gamma="scale", kernel="rbf", probability=True, random_state=44)
        simple.fit(x_tr, y_tr)
        simple_loss = model_log_loss(simple, x_te, y_te, labels)
        return simple_loss, simple_loss, simple_loss, {"C": 1.0, "gamma": "scale", "kernel": "rbf"}
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=44)
    base = SVC(probability=True, random_state=44)
    grid = {
        "C": [0.1, 1.0, 10.0],
        "gamma": ["scale", 0.1, 1.0],
        "kernel": ["rbf"]
    }
    grid_search = GridSearchCV(base, grid, scoring="neg_log_loss", cv=cv, n_jobs=1)
    grid_search.fit(x_tr, y_tr)

    random_grid = {
        "C": np.logspace(-2, 2, 12),
        "gamma": np.logspace(-3, 1, 12),
        "kernel": ["rbf"]
    }
    random_search = RandomizedSearchCV(base, random_grid, n_iter=8, scoring="neg_log_loss", cv=cv, random_state=44, n_jobs=1)
    random_search.fit(x_tr, y_tr)

    candidates = []
    for params in grid_search.cv_results_["params"]:
        candidates.append(params)
    stage_scores = []
    for params in candidates:
        model = SVC(probability=True, random_state=44, **params)
        score = cross_val_score(model, x_tr, y_tr, scoring="neg_log_loss", cv=2).mean()
        stage_scores.append(score)
    keep = np.argsort(stage_scores)[-3:]
    finalist_losses = []
    for idx in keep:
        model = SVC(probability=True, random_state=44, **candidates[idx])
        model.fit(x_tr, y_tr)
        finalist_losses.append(model_log_loss(model, x_te, y_te, labels))
    best_model = grid_search.best_estimator_
    grid_loss = model_log_loss(best_model, x_te, y_te, labels)
    random_loss = model_log_loss(random_search.best_estimator_, x_te, y_te, labels)
    staged_loss = min(finalist_losses)
    return grid_loss, random_loss, staged_loss, grid_search.best_params_

rows = []
metrics = []
search_results = []
for idx, (name, X, y) in enumerate(clf_ladder(), 1):
    grid_loss, random_loss, staged_loss, params = search_experiment(X, y)
    best_loss = min(grid_loss, random_loss, staged_loss)
    metrics.append(best_loss)
    search_results.append((name, grid_loss, random_loss, staged_loss, params))
    rows.append([f"D{idx}", f"{grid_loss:.3f}", f"{random_loss:.3f}", f"{staged_loss:.3f}", str(params)])
print_table(rows, ["rung", "grid", "random", "staged", "grid_best"])


## Results visualization
The first five panels preview the ladder data; the last panel tracks the metric from D1 to D5.

In [ ]:

plot_summary([r[0] for r in search_results], metrics, "best searched log loss vs rung", "log loss")


## Pitfall on the hardest rung
D5 is where a shortcut can look most convincing. The cell reproduces the wrong behavior and then applies the safer fix.

In [ ]:

x_tr, x_te, y_tr, y_te = scaled_train_test(clf_ladder()[-1][1], clf_ladder()[-1][2])
labels = np.unique(y_tr)
overfit_grid = []
for C in [0.1, 1.0, 10.0, 100.0]:
    model = SVC(C=C, gamma=1.0, probability=True, random_state=44)
    model.fit(x_tr, y_tr)
    train_loss = model_log_loss(model, x_tr, y_tr, labels)
    test_loss = model_log_loss(model, x_te, y_te, labels)
    cost = 0.010 * math.log10(C + 1.0)
    guarded = test_loss + cost
    overfit_grid.append((C, train_loss, test_loss, guarded))
rows = [[str(C), f"{tr:.3f}", f"{te:.3f}", f"{gu:.3f}"] for C, tr, te, gu in overfit_grid]
print_table(rows, ["C", "train_raw", "heldout", "heldout_plus_cost"])
raw_winner = min(overfit_grid, key=lambda row: row[1])
guarded_winner = min(overfit_grid, key=lambda row: row[3])
print(f"raw training winner C={raw_winner[0]} versus guarded winner C={guarded_winner[0]}")
assert raw_winner[0] >= guarded_winner[0]



## Evaluate it + Practice
- Compare the reported metric with a no-skill baseline before trusting the method.
- Run a cheap sanity check: shuffle labels or remove the key idea and confirm performance drops.
- Ablate the technique in the table above and inspect whether D5 changes more than D1.
- Watch failure signals: unstable validation numbers, suspiciously perfect scores, or a gap that grows with complexity.

Practice prompts:
1. Change one hyperparameter or preprocessing choice and rerun the ladder.


2. Add a small amount of label noise to D3 and explain which metric moved first.

3. On D5, write one sentence explaining whether the method is reducing bias, variance, or evaluation error.